# Functions

In [28]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm, uniform
from collections.abc import Callable
import plotly.graph_objects as go
N = 31851



def exp( x , a, A, tau):
    return a*N*expon.cdf(x , A , tau) 

def exp_for_variable_dataset(N: int):
    def exp( x , a, A, tau):
        return a*N*expon.cdf(x , A , tau) 
    return exp


    


def exp_unif(x, a, b, tau, A):
    return a * N * (expon.cdf(x, A, tau) + b * N * uniform.cdf(x, 0, 8e-6))

# def exp_gauss_cdf(N_exp , A , tau):
#     def fixed_exp( x , N , mu , sigma):
#         return exp( x , N_exp , A , tau) + gauss( x , N , mu , sigma)
#     return fixed_exp
    
# def gauss_exp_cdf(N_g , mu , sigma):
#     def fixed_gauss( x , N , A , tau):
#         return exp( x , N , A , tau) + gauss( x , N_g , mu , sigma)
#     return fixed_gauss

def exp_fit(cdf, a, A, tau, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, A, tau )
    n.fixed["A"] = True
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def exp_unif_fit(cdf, a, b, tau, A, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, b, tau, A )
    # n.fixed['N'] = True
    n.fixed['A'] = True
    # n.limits['b'] = (0, 1)
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , mu = mu , sigma = sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n




def dataset_analysis(dataset: np.ndarray , creator: Callable, bins: int , args: dict) -> Minuit:
    exp = creator(len(dataset))
    count, edges = np.histogram( dataset , bins=bins) 
    cost = ExtendedBinnedNLL(count, edges, exp)

    
    

    minuit_element =  Minuit(cost, *args.values())
    
    if "A" in args.keys():
        minuit_element.fixed["A"] = True

    return minuit_element

def migrad_hesse_display(m:Minuit) -> None:
    m.migrad()
    m.hesse()
    display(m)




    

# Dataset

In [29]:
data_0 = np.genfromtxt("Data/timestamp/21_01_2024_17_42.csv", delimiter=',')
data_1 = np.genfromtxt("Data/timestamp/23_01_2026_17_31.csv", delimiter=',')
data_2 = np.genfromtxt("Data/timestamp/29_01_2026_17_20.csv", delimiter=',')
data_3 = np.genfromtxt("Data/timestamp/02_02_2026_17_14.csv", delimiter=',')
data_4 = np.genfromtxt("Data/timestamp/05_02_2026_18_17.csv", delimiter=',')
data = np.concatenate((data_0, data_1, data_2, data_3, data_4 ))


# Variable/Constants

In [30]:
n_bins = 300
TAU_MU = 2.1969811e-6
exp_args = {"a": 1, "A": 0, "tau": TAU_MU}

# Analysis

## Hists

In [31]:

# data_2 = data_2[(data_2 > 0.6e-6) & (data_2 < 1.8e-6)]
count, edges = np.histogram(data, bins=n_bins , density=False)
count_1, edges_1 = np.histogram(data_1, bins=n_bins , density=False)
count_2, edges_2 = np.histogram(data_2, bins=n_bins , density=False)
count_3, edges_3 = np.histogram(data_3, bins=n_bins , density=False)
count_4, edges_4 = np.histogram(data_4, bins=n_bins , density=False)
print (len(data), len(data_1))
fig = go.Figure()

fig.add_trace(go.Bar(x=edges[:-1],  y=count   / len(data),   name='total',        width=np.diff(edges)))
fig.add_trace(go.Bar(x=edges_1[:-1], y=count_1 / len(data_1), name='23/01/2026',   width=np.diff(edges_1)))
fig.add_trace(go.Bar(x=edges_2[:-1], y=count_2 / len(data_2), name='29/01/2026',   width=np.diff(edges_2)))
fig.add_trace(go.Bar(x=edges_3[:-1], y=count_3 / len(data_3), name='02/02/2026',   width=np.diff(edges_3)))
fig.add_trace(go.Bar(x=edges_4[:-1], y=count_4 / len(data_4), name='05/02/2026',   width=np.diff(edges_4)))
fig.update_layout(
    xaxis_title='Time (s)',
    yaxis_title='Counts',
    bargap=0
)

fig.show()

31851 9400


In [32]:
k = dataset_analysis( data_1 , exp_for_variable_dataset , 69 , exp_args)
migrad_hesse_display(k)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 216.6 (χ²/ndof = 3.2)      │              Nfcn = 38               │
│ EDM = 1.03e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   1.027   │   0.011   │            │            │         │         │       │
│ 1 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
│ 2 │ tau  │ 2.230e-6  │ 0.029e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │           a           A         tau │
├─────┼─────────────────────────────────────┤
│   a │    0.000114           0 37.2538e-12 │
│   A │           0           0           0 │
│ tau │ 37.2538e-12           0    8.32e-16 │
└─────┴─────────────────────────────────────┘

## Full interpolation 

In [ ]:
n_bins = 100
exp_args = {"a": 1, "A": 0, "tau": TAU_MU}

data1 = data[(data < 7.1e-6)]## forse si può anche non tagliare dal basso


n = dataset_analysis( data1 , exp_for_variable_dataset , bins = n_bins , args=exp_args)
migrad_hesse_display(n)


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 176.5 (χ²/ndof = 1.8)      │              Nfcn = 46               │
│ EDM = 8.55e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   1.050   │   0.006   │            │            │         │         │       │
│ 1 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
│ 2 │ tau  │ 2.323e-6  │ 0.018e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬────────────────────────────────────────┐
│     │            a            A          tau │
├─────┼────────────────────────────────────────┤
│   a │     3.64e-05            0 22.67055e-12 │
│   A │            0            0            0 │
│ tau │ 22.67055e-12            0     3.32e-16 │
└─────┴────────────────────────────────────────┘

In [34]:
print(len(data_2),'\n', 'rate di decadimenti muonici tra il 29genn-2febbr:', 1/(len(data_2)/((24*4)*3600)), 'Hz')
print (len(data_3),'\n', 'rate di decadimenti muonici tra il 2-5 febbraio:', 1/(len(data_3)/((24*3+1)*3600)), 'Hz')
print (len(data_4),'\n', 'rate di decadimenti muonici tra il 5-11 febbraio:', 1/(len(data_4)/((24*4+20)*3600)), 'Hz')

6139 
 rate di decadimenti muonici tra il 29genn-2febbr: 56.29581365043167 Hz
4577 
 rate di decadimenti muonici tra il 2-5 febbraio: 57.4175223945816 Hz
8759 
 rate di decadimenti muonici tra il 5-11 febbraio: 47.67667541956844 Hz
